# Tutorial 4: Galactic Priors and Models

This tutorial explores the prior probability distributions used in brutus for modeling stars in the Milky Way context.

## Topics Covered

1. **IMF priors** for stellar masses
2. **Galactic structure priors** (thin disk, thick disk, halo)
3. **3D dust priors** with Bayestar
4. **Distance and parallax priors**
5. **Prior factorization** and combination

## Prerequisites

This tutorial requires the following brutus data files:
- `nn_c3k.h5` - Neural network for bolometric corrections
- `MIST_1.2_iso_vvcrit0.0.h5` - MIST isochrones
- `bayestar2019_v1.h5` (optional) - Bayestar dust map

If you don't have these files, run the optional download cell below.

In [ ]:
# Optional: Download required data files (only needed if not already cached)
# Uncomment the lines below to download them.

# from brutus.data import fetch_isos, fetch_nns, fetch_dustmaps
# fetch_isos()       # ~200 MB -- MIST isochrones
# fetch_nns()        # ~50 MB  -- Neural network for bolometric corrections
# fetch_dustmaps()   # ~2 GB   -- Bayestar 3D dust map (optional)

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tutorial_utils import (
    setup_tutorial,
    find_brutus_data_file,
    save_figure as _save_fig,
    print_section,
)

info = setup_tutorial(4, title="Tutorial 04: Galactic Priors and Models")
plots_dir = info['plot_dir']


def save_figure(fig, name):
    """Save figure to this tutorial's plot directory."""
    _save_fig(fig, 4, name)

## Section 2: Galactic Structure - 3D Density Models

The Galaxy has distinct structural components with different spatial distributions, ages, and metallicities.

### Components

- **Thin Disk**: Scale height ~300 pc, young/metal-rich stars
- **Thick Disk**: Scale height ~900 pc, old/metal-poor stars  
- **Halo**: Power-law profile, very old/metal-poor stars

Each component dominates at different Galactic latitudes and distances.

In [ ]:
from brutus.priors.galactic import logp_galactic_structure, logn_disk, logn_halo

# Create Galactic structure visualization  
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

# Define sightlines
sightlines = [
    (0, 90, 'North Galactic Pole'),
    (0, 0, 'Galactic Center'),
    (180, 0, 'Galactic Anti-center'),
    (90, 0, 'Perpendicular in-plane'),
    (45, 45, 'Intermediate'),
    (270, -30, 'South intermediate')
]

# Distance grid
distances = np.logspace(-2, 2, 200)  # 0.01 to 100 kpc

print("Computing Galactic priors for different sightlines...")

for idx, (l, b, title) in enumerate(sightlines[:6]):
    ax = axes[idx // 3, idx % 3]
    
    coord = np.array([l, b])
    
    # Get prior
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lnp = logp_galactic_structure(distances, coord)
    
    # Convert to cylindrical coordinates for component visualization
    from astropy.coordinates import SkyCoord, CylindricalRepresentation as CylRep
    import astropy.units as units
    
    ell = np.full_like(distances, l)
    b_arr = np.full_like(distances, b)
    coords = SkyCoord(l=ell * units.deg, b=b_arr * units.deg,
                     distance=distances * units.kpc, frame='galactic')
    coords_cyl = coords.galactocentric.cartesian.represent_as(CylRep)
    R, Z = coords_cyl.rho.value, coords_cyl.z.value
    
    # Compute individual components
    vol_factor = 2 * np.log(distances + 1e-300)
    lnp_thin = logn_disk(R, Z, R_scale=2.6, Z_scale=0.3) + vol_factor
    lnp_thick = logn_disk(R, Z, R_scale=2.0, Z_scale=0.9) + vol_factor + np.log(0.04)
    lnp_halo = logn_halo(R, Z) + vol_factor + np.log(0.005)
    
    # Convert to probabilities
    p_total = np.exp(lnp - np.max(lnp))
    p_thin = np.exp(lnp_thin - np.max(lnp))
    p_thick = np.exp(lnp_thick - np.max(lnp))
    p_halo = np.exp(lnp_halo - np.max(lnp))
    
    # Plot components
    ax.fill_between(distances, p_total, alpha=0.3, color='black', label='Total')
    ax.loglog(distances, p_thin, 'b-', lw=2, alpha=0.7, label='Thin Disk')
    ax.loglog(distances, p_thick, 'g-', lw=2, alpha=0.7, label='Thick Disk')
    ax.loglog(distances, p_halo, 'r-', lw=2, alpha=0.7, label='Halo')
    
    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('Relative Probability')
    ax.set_title(f'{title}\n(l={l}, b={b})')
    ax.set_xlim(0.01, 100)
    ax.set_ylim(1e-6, 2)
    ax.grid(True, alpha=0.3)
    
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

# Panel 7: Metallicity distributions
ax = axes[2, 0]

feh_range = np.linspace(-3, 0.5, 200)

# Component metallicity distributions (schematic)
thin_feh = np.exp(-(feh_range + 0.1)**2 / (2 * 0.2**2))
thick_feh = np.exp(-(feh_range + 0.6)**2 / (2 * 0.3**2))
halo_feh = np.exp(-(feh_range + 1.5)**2 / (2 * 0.5**2))

ax.plot(feh_range, thin_feh/thin_feh.max(), 'b-', lw=2, label='Thin Disk')
ax.plot(feh_range, thick_feh/thick_feh.max(), 'g-', lw=2, label='Thick Disk')
ax.plot(feh_range, halo_feh/halo_feh.max(), 'r-', lw=2, label='Halo')

ax.set_xlabel('[Fe/H]')
ax.set_ylabel('Relative Probability')
ax.set_title('Metallicity Distributions (schematic)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 8: Age distributions
ax = axes[2, 1]

age_range = np.linspace(0, 14, 200)  # Gyr

thin_age = np.exp(-(age_range - 4)**2 / (2 * 3**2)) * (age_range < 10)
thick_age = np.exp(-(age_range - 10)**2 / (2 * 1.5**2)) * (age_range > 8)
halo_age = np.exp(-(age_range - 12)**2 / (2 * 1**2)) * (age_range > 10)

ax.plot(age_range, thin_age/thin_age.max(), 'b-', lw=2, label='Thin Disk')
ax.plot(age_range, thick_age/thick_age.max(), 'g-', lw=2, label='Thick Disk')
ax.plot(age_range, halo_age/halo_age.max(), 'r-', lw=2, label='Halo')

ax.set_xlabel('Age (Gyr)')
ax.set_ylabel('Relative Probability')
ax.set_title('Age Distributions (schematic)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 9: Parameters text
ax = axes[2, 2]
ax.axis('off')

components_info = """
Galactic Component Parameters:

Thin Disk:
  Scale height: 300 pc
  Scale length: 2.6 kpc
  Solar density: 0.04 M_sun/pc^3

Thick Disk:
  Scale height: 900 pc
  Scale length: 3.6 kpc
  Solar density: 0.0025 M_sun/pc^3

Halo:
  Core radius: 2.0 kpc
  Power law: r^(-3.39)
  Solar density: 0.00015 M_sun/pc^3
"""

ax.text(0.05, 0.95, components_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Galactic Structure Priors', fontsize=16, fontweight='bold')
save_figure(fig, 'galactic_structure')
plt.show()

print("\nGalactic structure prior demonstrations complete")
print("  Key points:")
print("  - Thin disk dominates at low latitudes")
print("  - Halo becomes important at high latitudes and large distances")
print("  - Each component has distinct [Fe/H] and age distributions")

## Section 3: 3D Dust Extinction

The `Bayestar` class in `brutus.dust` provides distance-resolved extinction estimates. For any Galactic sightline, it returns A(V) mean and standard deviation as a function of distance, based on Pan-STARRS and 2MASS photometry.

Key properties:
- 31 distance bins from ~0.06 to ~60 kpc at ~7 arcmin HEALPix resolution
- A(V) uncertainty grows with distance
- Extinction concentrated near the Galactic plane
- Returns uniform prior for sightlines outside map coverage

In [ ]:
# 3D Dust extinction demonstration using the Bayestar dust map
#
# The Bayestar class provides distance-resolved extinction profiles:
#   distances, av_mean, av_std = dustmap.query(coord)
# Each sightline returns A(V) as a function of distance.

# Try to load the real Bayestar dust map
dustmap = None
try:
    from brutus.dust import Bayestar
    dustfile = find_brutus_data_file('bayestar2019_v1.h5')
    dustmap = Bayestar(dustfile=dustfile)
    print(f"Loaded Bayestar dust map from {dustfile}")
    dust_available = True
except (FileNotFoundError, ImportError) as e:
    print(f"Bayestar dust map not available: {e}")
    print("  Using synthetic data for illustration.")
    dust_available = False

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Define sightlines for dust queries
sightlines_dust = [
    (0, 0, 'Galactic Center', 'red'),
    (90, 0, 'l=90, b=0', 'blue'),
    (180, 0, 'Anti-center', 'green'),
    (45, 30, 'l=45, b=30', 'orange'),
    (0, 90, 'North Pole', 'purple'),
]

# Panel 1: A(V) vs distance for different sightlines
ax = axes[0, 0]

if dust_available:
    from astropy.coordinates import SkyCoord
    import astropy.units as units

    print("\nQuerying Bayestar dust map for different sightlines...")

    for l, b, label, color in sightlines_dust:
        coord = SkyCoord(l=l * units.deg, b=b * units.deg, frame='galactic')
        dists, av_mean, av_std = dustmap.query(coord)

        finite = np.isfinite(av_mean)
        if np.any(finite):
            ax.plot(dists[finite], av_mean[finite], color=color, lw=2,
                    alpha=0.7, label=label)
            ax.fill_between(dists[finite],
                            (av_mean - av_std)[finite],
                            (av_mean + av_std)[finite],
                            alpha=0.15, color=color)
            print(f"  (l={l:3d}, b={b:+3d}): max A(V) = {np.nanmax(av_mean):.2f}")

    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('A(V) (mag)')
    ax.set_title('Bayestar Extinction Profiles')
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
else:
    # Synthetic fallback
    distances_synth = np.logspace(-1, 1.5, 100)
    for l, b, label, color in sightlines_dust:
        scale = np.exp(-abs(b) / 15.0)
        av_synth = 1.5 * scale * (1 - np.exp(-distances_synth / 2.0))
        ax.plot(distances_synth, av_synth, color=color, lw=2, alpha=0.7, label=label)
    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('A(V) (mag)')
    ax.set_title('Extinction vs Distance (synthetic)')
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Panel 2: Extinction uncertainty growth with distance
ax = axes[0, 1]

if dust_available:
    coord = SkyCoord(l=90 * units.deg, b=0 * units.deg, frame='galactic')
    dists, av_mean, av_std = dustmap.query(coord)

    finite = np.isfinite(av_mean)
    if np.any(finite):
        ax.fill_between(dists[finite],
                        (av_mean - av_std)[finite],
                        (av_mean + av_std)[finite],
                        alpha=0.3, color='blue', label='1-sigma')
        ax.fill_between(dists[finite],
                        (av_mean - 2 * av_std)[finite],
                        (av_mean + 2 * av_std)[finite],
                        alpha=0.15, color='blue', label='2-sigma')
        ax.plot(dists[finite], av_mean[finite], 'b-', lw=2, label='Mean A(V)')

    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('A(V) (mag)')
    ax.set_title('Extinction Uncertainty (l=90, b=0)')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    distances_synth = np.logspace(-1, 1.5, 50)
    av_mean_synth = 0.5 * (1 - np.exp(-distances_synth / 3.0))
    av_std_synth = 0.1 * av_mean_synth * (distances_synth / 5.0)
    ax.fill_between(distances_synth, av_mean_synth - av_std_synth,
                    av_mean_synth + av_std_synth, alpha=0.3, color='blue',
                    label='1-sigma')
    ax.plot(distances_synth, av_mean_synth, 'b-', lw=2, label='Mean A(V)')
    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('A(V) (mag)')
    ax.set_title('Extinction Uncertainty (synthetic)')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Panel 3: A(V) distribution across sightlines at a fixed distance
ax = axes[0, 2]

if dust_available:
    rng = np.random.default_rng(42)
    l_sample = rng.uniform(0, 360, 500)
    b_sample = rng.uniform(-90, 90, 500)
    av_at_1kpc = []

    for ls, bs in zip(l_sample, b_sample):
        coord = SkyCoord(l=ls * units.deg, b=bs * units.deg, frame='galactic')
        dists, av_mean, av_std = dustmap.query(coord)
        idx_1kpc = np.argmin(np.abs(dists - 1.0))
        if np.isfinite(av_mean[idx_1kpc]):
            av_at_1kpc.append(av_mean[idx_1kpc])

    av_at_1kpc = np.array(av_at_1kpc)
    av_at_1kpc = av_at_1kpc[av_at_1kpc < 5]

    ax.hist(av_at_1kpc, bins=50, density=True, alpha=0.7, color='green',
            edgecolor='white', linewidth=0.3)
    ax.axvline(np.median(av_at_1kpc), color='red', ls='--', lw=2,
               label=f'Median = {np.median(av_at_1kpc):.2f}')
    ax.set_title('A(V) at ~1 kpc (500 sightlines)')
else:
    av_samples = np.random.gamma(2, 0.2, 10000)
    av_samples = av_samples[av_samples < 5]
    ax.hist(av_samples, bins=50, density=True, alpha=0.7, color='green',
            edgecolor='white', linewidth=0.3)
    ax.axvline(np.median(av_samples), color='red', ls='--', lw=2,
               label=f'Median = {np.median(av_samples):.2f}')
    ax.set_title('A(V) Distribution (synthetic)')

ax.set_xlabel('A(V) (mag)')
ax.set_ylabel('Probability Density')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Latitude dependence of extinction
ax = axes[1, 0]

if dust_available:
    b_values = np.arange(-80, 81, 10)
    av_vs_b = []
    for b_val in b_values:
        coord = SkyCoord(l=90 * units.deg, b=b_val * units.deg, frame='galactic')
        dists, av_mean, av_std = dustmap.query(coord)
        idx_2kpc = np.argmin(np.abs(dists - 2.0))
        av_vs_b.append(av_mean[idx_2kpc] if np.isfinite(av_mean[idx_2kpc]) else np.nan)

    ax.plot(b_values, av_vs_b, 'ko-', lw=2, markersize=5)
    ax.set_title('Latitude Dependence (l=90, d=2 kpc)')
else:
    b_values = np.linspace(-80, 80, 50)
    av_vs_b = 1.0 * np.exp(-np.abs(b_values) / 10.0)
    ax.plot(b_values, av_vs_b, 'ko-', lw=2, markersize=3)
    ax.set_title('Latitude Dependence (synthetic)')

ax.set_xlabel('Galactic Latitude b (deg)')
ax.set_ylabel('A(V) (mag)')
ax.grid(True, alpha=0.3)

# Panel 5: R(V) prior
ax = axes[1, 1]

rv_values = np.linspace(2.0, 6.0, 100)
rv_prob = np.exp(-(rv_values - 3.1)**2 / (2 * 0.3**2))
ax.plot(rv_values, rv_prob / rv_prob.max(), 'b-', lw=2)
ax.axvline(3.1, color='red', ls='--', lw=2, label='Standard R(V) = 3.1')
ax.fill_between(rv_values, rv_prob / rv_prob.max(), alpha=0.3, color='blue')

ax.set_xlabel('R(V)')
ax.set_ylabel('Relative Probability')
ax.set_title('R(V) Prior')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 6: Summary text
ax = axes[1, 2]
ax.axis('off')

dust_info = """
3D Dust Extinction in brutus:

Bayestar Dust Map:
  Resolution: ~7 arcmin (HEALPix)
  Distance bins: 31 (0.06-60 kpc)
  Provides A(V) mean and std
  Based on Pan-STARRS + 2MASS

Key Properties:
  Higher extinction near plane
  Uncertainty grows with distance
  Returns uniform prior outside
    map coverage

Usage in BruteForce:
  dustfile parameter in fit()
  Prior on A(V) given (l,b,d)
  Marginalized during fitting

R(V) Variation:
  Standard: R(V) = 3.1
  Dense clouds: R(V) ~ 2.5
  Diffuse ISM: R(V) ~ 5.0
"""

ax.text(0.05, 0.95, dust_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('3D Dust Extinction', fontsize=16, fontweight='bold')
save_figure(fig, 'dust_extinction')
plt.show()

print("\n3D dust extinction demonstrations complete")
print("  Key points:")
print("  - Extinction increases with distance")
print("  - Higher extinction near Galactic plane")
print("  - Uncertainties grow with distance")

## Section 4: Parallax and Distance Priors

Parallax measurements from Gaia constrain stellar distances. The `logp_parallax` function provides a Gaussian log-prior on the model parallax given an observed parallax and its uncertainty.

Key concepts:
- **Non-linear transformation**: d = 1/pi leads to asymmetric distance uncertainties
- **Lutz-Kelker bias**: Volume effects bias distance estimates outward at low signal-to-noise
- **Scale factors**: brutus works internally with s ~ pi^2 for flux-based fitting (see Section 7)

In [ ]:
from brutus.priors.astrometric import logp_parallax

# Parallax/distance visualization (2x2)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Parallax-distance transformation
ax = axes[0, 0]

parallaxes = np.linspace(0.1, 10, 1000)  # mas
distances_px = 1.0 / parallaxes  # kpc

ax.plot(parallaxes, distances_px, 'b-', lw=2)
ax.fill_between(parallaxes, distances_px * 0.9, distances_px * 1.1,
                alpha=0.3, color='blue', label='+/-10% uncertainty')

ax.set_xlabel('Parallax (mas)')
ax.set_ylabel('Distance (kpc)')
ax.set_title('Parallax-Distance Relation')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.legend()
ax.grid(True, alpha=0.3)

for pi, d in [(10, 0.1), (1, 1), (0.1, 10)]:
    ax.plot(pi, d, 'ro', markersize=8)
    ax.annotate(f'{pi} mas\n{d} kpc', (pi, d),
               xytext=(5, 5), textcoords='offset points', fontsize=8)

# Panel 2: logp_parallax for different true distances
ax = axes[0, 1]

target_distances = [0.5, 1.0, 2.0, 5.0]  # kpc
colors = ['blue', 'green', 'orange', 'red']
parallax_grid = np.linspace(0.01, 5, 500)

for d, color in zip(target_distances, colors):
    true_pi = 1.0 / d
    sigma_pi = 0.05

    lnp = logp_parallax(parallax_grid, true_pi, sigma_pi)
    p = np.exp(lnp - np.max(lnp))

    ax.plot(parallax_grid, p, color=color, lw=2, label=f'd = {d} kpc')

ax.set_xlabel('Observed Parallax (mas)')
ax.set_ylabel('Relative Probability')
ax.set_title('logp_parallax for Different Distances')
ax.set_xlim(0, 3)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Effect of parallax uncertainty
ax = axes[1, 0]

parallax_grid2 = np.linspace(-0.5, 3.0, 500)
p_meas = 1.0

for sigma, color, label in [(0.02, 'blue', 'sig=0.02 (S/N=50)'),
                              (0.1, 'green', 'sig=0.1 (S/N=10)'),
                              (0.25, 'orange', 'sig=0.25 (S/N=4)'),
                              (0.5, 'red', 'sig=0.5 (S/N=2)')]:
    lnp = logp_parallax(parallax_grid2, p_meas, sigma)
    p = np.exp(lnp - np.max(lnp))
    ax.plot(parallax_grid2, p, lw=2, color=color, label=label)

ax.axvline(p_meas, color='black', ls='--', lw=1, alpha=0.5, label='Measured')
ax.set_xlabel('Model Parallax (mas)')
ax.set_ylabel('Relative Probability')
ax.set_title('logp_parallax: Effect of Uncertainty')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 4: Summary text
ax = axes[1, 1]
ax.axis('off')

parallax_info = """
Parallax & Distance Priors:

Key Relations:
  d = 1/pi (kpc for pi in mas)
  sig_d/d ~ sig_pi/pi (small errors)
  Asymmetric for large errors

logp_parallax in brutus:
  Gaussian prior on parallax
  Returns 0 (flat) if invalid
  Used in BruteForce fitting

logp_parallax_scale:
  Works with scale factors s~pi^2
  For direct flux comparison
  See next section for details
"""

ax.text(0.05, 0.95, parallax_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Parallax and Distance Priors', fontsize=14, fontweight='bold')
fig.tight_layout()
save_figure(fig, 'parallax_distance')
plt.show()

print("\nParallax and distance prior demonstrations complete")
print("  Key points:")
print("  - logp_parallax provides Gaussian prior on observed parallax")
print("  - Non-linear parallax-distance transformation causes asymmetric errors")
print("  - Width of prior controlled by measurement uncertainty")

## Section 5: Prior Factorization and Combination

The complete prior in brutus combines all components multiplicatively, with proper factorization based on conditional independence.

### Prior Factorization

The full prior can be written as:

P(θ) = P(M) × P(d,Z,τ|l,b) × P(A_V|d,l,b) × P(π_obs|d)

Where:
- P(M): IMF prior on stellar mass
- P(d,Z,τ|l,b): Galactic structure prior (distance, metallicity, age given position)
- P(A_V|d,l,b): 3D dust prior (extinction given distance and position)
- P(π_obs|d): Parallax likelihood (observed parallax given true distance)

In [ ]:
# Demonstrate prior combination
# Note: We combine individual prior components manually, as brutus
# applies them internally during BruteForce fitting.

from brutus.priors.stellar import logp_imf

# Create combined prior visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Prior components along a sightline
ax = axes[0, 0]

distances = np.logspace(-1, 1.5, 500)  # finer grid for smooth curves
l, b = 90, 30  # Intermediate latitude
coord = np.array([l, b])

# Compute individual prior components using real brutus functions

# Galactic structure prior
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lnp_gal = logp_galactic_structure(distances, coord)

# Dust (simplified model -- full Bayestar gives per-sightline profiles)
av_mean = 0.3 * (1 - np.exp(-distances / 3.0))
lnp_dust = -0.5 * (av_mean / 0.2)**2

# Parallax constraint using brutus logp_parallax
# Use moderate uncertainty so all three components contribute visibly
obs_parallax = 0.5    # mas (=> d ~ 2 kpc)
sigma_parallax = 0.2  # mas (S/N ~ 2.5, broad prior in distance space)
true_parallax = 1.0 / distances
lnp_par = logp_parallax(true_parallax, obs_parallax, sigma_parallax)

# Total prior
lnp_total = lnp_gal + lnp_dust + lnp_par

# Convert to probabilities for plotting
p_gal = np.exp(lnp_gal - np.max(lnp_gal))
p_dust = np.exp(lnp_dust - np.max(lnp_dust))
p_parallax = np.exp(lnp_par - np.max(lnp_par))
p_total = np.exp(lnp_total - np.max(lnp_total))

ax.plot(distances, p_gal, 'b-', lw=2, alpha=0.7, label='Galactic')
ax.plot(distances, p_dust, 'g-', lw=2, alpha=0.7, label='Dust')
ax.plot(distances, p_parallax, 'r-', lw=2, alpha=0.7, label='Parallax')
ax.plot(distances, p_total, 'k-', lw=3, label='Total')

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('Relative Probability')
ax.set_title(f'Prior Components (l={l}, b={b}, pi={obs_parallax}+/-{sigma_parallax} mas)')
ax.set_xscale('log')
ax.set_xlim(0.1, 30)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: 2D prior surface (distance vs extinction)
ax = axes[0, 1]

d_grid = np.logspace(-0.5, 1, 50)
av_grid = np.linspace(0, 3, 50)
D, AV = np.meshgrid(d_grid, av_grid)

# Compute 2D prior (simplified Galactic + dust model for illustration)
lnp_2d = np.zeros_like(D)
for i in range(len(d_grid)):
    for j in range(len(av_grid)):
        lnp_gal_ij = -0.5 * ((D[j, i] - 1.0) / 2.0)**2
        av_expected = 0.3 * (1 - np.exp(-D[j, i] / 3.0))
        lnp_dust_ij = -0.5 * ((AV[j, i] - av_expected) / 0.2)**2
        lnp_2d[j, i] = lnp_gal_ij + lnp_dust_ij

p_2d = np.exp(lnp_2d - np.max(lnp_2d))
contours = ax.contourf(D, AV, p_2d, levels=20, cmap='viridis')
ax.contour(D, AV, p_2d, levels=10, colors='white', alpha=0.3, linewidths=0.5)

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('2D Prior: Distance vs Extinction')
ax.set_xscale('log')
plt.colorbar(contours, ax=ax, label='Relative Probability')

# Panel 3: IMF prior using brutus logp_imf
ax = axes[1, 0]

masses = np.logspace(-1, 1.5, 100)

# Show IMF with different high-mass slopes
for alpha_h, color, label in [(1.8, 'blue', 'alpha_high=1.8 (top-heavy)'),
                                (2.3, 'green', 'alpha_high=2.3 (Kroupa)'),
                                (2.7, 'red', 'alpha_high=2.7 (steep)')]:
    lnp = logp_imf(masses, alpha_low=1.3, alpha_high=alpha_h, mass_break=0.5)
    p = np.exp(lnp - np.max(lnp))
    ax.loglog(masses, p, color=color, lw=2, alpha=0.8, label=label)

ax.axvline(0.5, color='gray', ls=':', alpha=0.5, label='Mass break (0.5 Msun)')
ax.set_xlabel('Initial Mass (Msun)')
ax.set_ylabel('Relative Probability')
ax.set_title('logp_imf: Initial Mass Function')
ax.set_xlim(0.1, 30)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 4: Prior combination summary
ax = axes[1, 1]
ax.axis('off')

combination_text = """
Prior Factorization in brutus:

Full Prior:
P(theta) = P(M) * P(d,Z,tau|l,b)
         * P(A_V|d,l,b) * P(pi|d)

Components:
1. P(M): IMF prior (logp_imf)
   - Broken power law (Kroupa)
   - Mass range: 0.08-100 Msun

2. P(d,Z,tau|l,b): Galactic structure
   - logp_galactic_structure
   - Thin disk, thick disk, halo

3. P(A_V|d,l,b): 3D dust
   - logp_extinction + Bayestar
   - Distance-dependent

4. P(pi|d): Parallax constraint
   - logp_parallax / logp_parallax_scale
   - Gaia measurements

5. P([Fe/H]): Metallicity (logp_feh)
6. P(age|[Fe/H]): Age (logp_age_from_feh)
"""

ax.text(0.05, 0.95, combination_text, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Prior Combination and Factorization', fontsize=16, fontweight='bold')
save_figure(fig, 'prior_combination')
plt.show()

print("\nPrior combination demonstrations complete")
print("  Key points:")
print("  - Priors combine multiplicatively")
print("  - Each component provides independent constraints")
print("  - Total prior shapes posterior distribution")

## Section 6: Metallicity and Age-Metallicity Priors

Beyond Galactic structure, brutus provides priors on stellar metallicity and an age-metallicity relation that encodes the observed correlation between stellar age and chemical enrichment.

### Metallicity Prior: `logp_feh`

The metallicity prior is a simple Gaussian parameterized by a mean and dispersion, with typical values for each Galactic component:

- **Thin disk**: `feh_mean = -0.2`, `feh_sigma = 0.3`
- **Thick disk**: `feh_mean = -0.7`, `feh_sigma = 0.4`
- **Halo**: `feh_mean = -1.6`, `feh_sigma = 0.5`

### Age-Metallicity Relation: `logp_age_from_feh`

The age prior is conditioned on metallicity through a logistic age-metallicity relation: metal-poor stars are preferentially older, while metal-rich stars are preferentially younger. The mean age and its dispersion are both functions of `feh_mean`, with ages drawn from a truncated normal distribution bounded by physically reasonable limits.

In [ ]:
from brutus.priors.galactic import logp_feh, logp_age_from_feh

# --- Panel (a): [Fe/H] distribution for disk vs halo ---
feh_grid = np.linspace(-3, 0.5, 300)

# Typical parameters for Galactic components
components = {
    'Thin Disk':  dict(feh_mean=-0.2, feh_sigma=0.3, color='blue'),
    'Thick Disk': dict(feh_mean=-0.7, feh_sigma=0.4, color='green'),
    'Halo':       dict(feh_mean=-1.6, feh_sigma=0.5, color='red'),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, params in components.items():
    lnp = logp_feh(feh_grid, feh_mean=params['feh_mean'],
                   feh_sigma=params['feh_sigma'])
    prob = np.exp(lnp - np.max(lnp))
    ax1.plot(feh_grid, prob, lw=2, color=params['color'],
             label=f"{name} (mu={params['feh_mean']}, sig={params['feh_sigma']})")

ax1.set_xlabel('[Fe/H] (dex)')
ax1.set_ylabel('Relative Probability')
ax1.set_title('Metallicity Prior: logp_feh')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Panel (b): Age-metallicity relation for different [Fe/H] ---
age_grid = np.linspace(0.01, 14, 300)  # ages in Gyr

feh_values = [0.0, -0.2, -0.7, -1.6]
feh_colors = ['orange', 'blue', 'green', 'red']
feh_labels = ['Solar ([Fe/H]=0.0)', 'Thin Disk ([Fe/H]=-0.2)',
              'Thick Disk ([Fe/H]=-0.7)', 'Halo ([Fe/H]=-1.6)']

for feh_val, color, label in zip(feh_values, feh_colors, feh_labels):
    lnp = logp_age_from_feh(age_grid, feh_mean=feh_val)
    prob = np.exp(lnp - np.max(lnp))
    ax2.plot(age_grid, prob, lw=2, color=color, label=label)

ax2.set_xlabel('Age (Gyr)')
ax2.set_ylabel('Relative Probability')
ax2.set_title('Age Prior Conditioned on [Fe/H]: logp_age_from_feh')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle('Metallicity and Age-Metallicity Priors', fontsize=14,
             fontweight='bold')
fig.tight_layout()
save_figure(fig, 'feh_age_priors')
plt.show()

print("\nMetallicity & age-metallicity prior demonstrations complete")
print("  Key points:")
print("  - logp_feh is a Gaussian prior on [Fe/H] with component-specific means")
print("  - logp_age_from_feh encodes the age-metallicity relation:")
print("    metal-poor populations are preferentially older")
print("  - The age dispersion narrows for metal-rich (younger) populations")

## Section 7: Parallax Scale Factor and Conversion

When fitting stellar SEDs, brutus works internally with **flux density scale factors** rather than distances. The scale factor `s` is proportional to the square of the parallax (`s ~ pi^2`), so it controls the overall normalization of the observed fluxes relative to the model.

### Key Functions

- **`convert_parallax_to_scale(p_meas, p_err)`**: Converts a parallax measurement and its uncertainty into scale factor statistics `(s_mean, s_std)` via error propagation. For high signal-to-noise measurements (`|p_meas/p_err| > snr_lim`), the conversion uses `s_mean = max(0, p_meas)^2 + p_err^2` and propagates the uncertainty analytically. For low S/N, it returns uninformative (flat) statistics.

- **`logp_parallax_scale(scales, scale_errs, p_meas, p_err)`**: Evaluates the log-prior on scale factors given a parallax measurement. For high S/N parallaxes this is a Gaussian centered on the converted scale factor; for low S/N it returns a uniform (zero) prior.

In [ ]:
from brutus.priors.astrometric import logp_parallax_scale, convert_parallax_to_scale

# ---------------------------------------------------------------
# Demonstrate convert_parallax_to_scale and its Gaussian approximation
# ---------------------------------------------------------------
print("Parallax -> Scale factor conversion examples")
print("-" * 50)
example_cases = [
    (1.0, 0.02, "Nearby star, high S/N (pi/sig = 50)"),
    (1.0, 0.1,  "Nearby star, moderate S/N (pi/sig = 10)"),
    (0.5, 0.1,  "Intermediate distance, S/N = 5"),
    (0.2, 0.1,  "Distant star, low S/N (pi/sig = 2)"),
]
for p_meas, p_err, desc in example_cases:
    s_mean, s_std = convert_parallax_to_scale(p_meas, p_err)
    print(f"  pi = {p_meas:.2f} +/- {p_err:.2f} mas  =>  "
          f"s_mean = {s_mean:.4f}, s_std = {s_std:.4f}   ({desc})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ---------------------------------------------------------------
# Panel (a): Gaussian approximation vs true MC distribution
#
# convert_parallax_to_scale uses a 2nd-order Taylor expansion:
#   s_mean = pi^2 + sig_pi^2   (includes bias correction)
#   s_std  = sqrt(2*sig_pi^4 + 4*pi^2*sig_pi^2)
#
# At high S/N this matches well; at low S/N the true distribution
# of s = pi^2 is skewed, motivating the snr_lim threshold.
#
# Plot in standardized units (s - s_mean)/s_std so all S/N cases
# overlay on the same axes.
# ---------------------------------------------------------------

scenarios = [
    (2.0,  0.1, 'blue',   'S/N = 20'),
    (1.0,  0.1, 'green',  'S/N = 10'),
    (0.5,  0.1, 'orange', 'S/N = 5'),
    (0.25, 0.1, 'red',    'S/N = 2.5'),
]

rng = np.random.default_rng(42)

# Gaussian reference (standard normal)
x_std = np.linspace(-4, 6, 300)
gauss_std = np.exp(-0.5 * x_std**2) / np.sqrt(2 * np.pi)
ax1.plot(x_std, gauss_std, 'k--', lw=2.5, label='Gaussian approx', zorder=10)

for p_meas, p_err, color, label in scenarios:
    # True distribution via MC: s = pi^2, pi ~ N(p_meas, p_err)
    pi_samples = rng.normal(p_meas, p_err, 200000)
    s_samples = pi_samples**2

    # Error propagation statistics (same formula as convert_parallax_to_scale)
    s_mean = max(0, p_meas)**2 + p_err**2
    s_std = np.sqrt(2 * p_err**4 + 4 * p_meas**2 * p_err**2)

    # Standardize to (s - s_mean) / s_std
    s_standardized = (s_samples - s_mean) / s_std

    ax1.hist(s_standardized, bins=100, range=(-4, 6), density=True,
             alpha=0.35, color=color, label=label)

ax1.set_xlabel('Standardized: (s - s_mean) / s_std')
ax1.set_ylabel('Probability Density')
ax1.set_title('Gaussian Approx vs True Distribution of s = pi^2')
ax1.legend(fontsize=8)
ax1.set_xlim(-4, 6)
ax1.grid(True, alpha=0.3)

# ---------------------------------------------------------------
# Panel (b): Parallax to scale factor conversion curves
# ---------------------------------------------------------------
pi_grid = np.linspace(0.05, 3.0, 200)
p_errs_to_show = [0.02, 0.05, 0.1, 0.2]
colors_b = ['blue', 'green', 'orange', 'red']

for p_err, color in zip(p_errs_to_show, colors_b):
    s_means = []
    s_stds = []
    for p_val in pi_grid:
        sm, ss = convert_parallax_to_scale(p_val, p_err)
        s_means.append(sm)
        s_stds.append(ss)
    s_means = np.array(s_means)
    s_stds = np.array(s_stds)

    # Only plot the informative regime (above S/N threshold)
    snr = np.abs(pi_grid / p_err)
    mask = snr > 4.0

    ax2.plot(pi_grid[mask], s_means[mask], lw=2, color=color,
             label=f'sig_pi = {p_err} mas')
    ax2.fill_between(pi_grid[mask],
                     s_means[mask] - s_stds[mask],
                     s_means[mask] + s_stds[mask],
                     alpha=0.15, color=color)

# Overlay the exact relation s = pi^2
pi_exact = np.linspace(0.05, 3.0, 200)
ax2.plot(pi_exact, pi_exact**2, 'k--', lw=1.5, alpha=0.6, label='s = pi^2 (exact)')

ax2.set_xlabel('Parallax (mas)')
ax2.set_ylabel('Scale Factor s')
ax2.set_title('Parallax to Scale Factor Conversion')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle('Parallax Scale Factor Prior and Conversion', fontsize=14,
             fontweight='bold')
fig.tight_layout()
save_figure(fig, 'parallax_scale_prior')
plt.show()

print("\nParallax scale factor demonstrations complete")
print("  Key points:")
print("  - convert_parallax_to_scale maps pi +/- sig_pi to (s_mean, s_std)")
print("  - Gaussian approximation works well at high S/N but becomes skewed at low S/N")
print("  - Below snr_lim (default 4), brutus returns a flat (uninformative) prior")
print("  - The bias term (s_mean = pi^2 + sig_pi^2) correctly captures E[pi^2]")

## Summary and Key Takeaways

This tutorial has covered the prior probability distributions used in brutus:

### Key Priors

1. **IMF**: Determines stellar mass distribution
   - Kroupa: Broken power law (standard choice)
   - Salpeter: Single power law
   - Affects M/L ratios and observable fractions

2. **Galactic Structure**: 3D spatial distribution
   - Thin disk: Young, metal-rich, low scale height
   - Thick disk: Old, metal-poor, high scale height  
   - Halo: Very old, very metal-poor, power-law

3. **Dust Maps**: 3D extinction (Bayestar)
   - Provides A(V) as function of distance
   - Higher extinction in Galactic plane
   - Uncertainties increase with distance

4. **Parallax**: Distance constraints from Gaia
   - Non-linear parallax-distance transformation
   - Lutz-Kelker bias pushes distances outward
   - Systematic corrections needed

### Prior Factorization

The full prior combines multiplicatively:

P(θ) = P(M) × P(d,Z,τ|l,b) × P(A_V|d,l,b) × P(π_obs|d)

Components are independent given position and observables.

### Next Steps

- **Tutorial 5**: Fitting Individual Stars with BruteForce
- **Tutorial 6**: Cluster Analysis and Population Fitting
- **Tutorial 7**: 3D Dust Mapping

In [ ]:
print("Tutorial 4 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")